# 4. Results Analysis

**Purpose**: Understand results, identify issues, prepare final outputs.

## Sections
1. Load predictions
2. Match rate by source
3. Confidence distribution
4. Sample matches at each tier
5. Unmatched records analysis
6. Export final outputs


---
## 1. Setup and Load Data


In [ ]:
import sys
sys.path.insert(0, '/Users/robertlalani/Desktop/entity_resolution_12_18_25/01-05-26')

import pandas as pd
import numpy as np
from pathlib import Path

from config import config
from utils import (
    log_step, 
    load_checkpoint,
    save_json
)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 100)

print("Imports loaded successfully")


In [ ]:
# Load predictions
predictions_df = load_checkpoint(config.paths.PREDICTIONS, "Predictions")

if predictions_df is None:
    raise FileNotFoundError(f"Predictions not found at {config.paths.PREDICTIONS}. Run 3_inference.ipynb first.")

print(f"\nPREDICTIONS LOADED")
print("=" * 50)
print(f"Total predictions: {len(predictions_df):,}")
print(f"Columns: {list(predictions_df.columns)}")


In [ ]:
# Load reference data for name lookups
dim_org_df = load_checkpoint(config.paths.DATA_DIR + "/dim_org_training.parquet", "dim_org")

if dim_org_df is not None:
    # Create lookup dictionaries
    dim_org_names = dict(zip(dim_org_df['unique_id'], dim_org_df['name']))
    print(f"Loaded dim_org lookup: {len(dim_org_names):,} records")
else:
    dim_org_names = {}
    print("Warning: Could not load dim_org for name lookups")


---
## 2. Cluster Analysis


In [ ]:
# Load clusters
clusters_df = load_checkpoint(config.paths.CLUSTERS, "Entity clusters")

if clusters_df is not None:
    print("CLUSTER SIZE DISTRIBUTION")
    print("=" * 50)
    
    cluster_sizes = clusters_df.groupby('cluster_id').size()
    
    # Basic stats
    print(f"\nTotal clusters: {len(cluster_sizes):,}")
    print(f"Total records in clusters: {len(clusters_df):,}")
    
    # Size categories
    singleton = (cluster_sizes == 1).sum()
    size_2 = (cluster_sizes == 2).sum()
    size_3_5 = ((cluster_sizes >= 3) & (cluster_sizes <= 5)).sum()
    size_6_10 = ((cluster_sizes >= 6) & (cluster_sizes <= 10)).sum()
    size_large = (cluster_sizes > 10).sum()
    
    print(f"\nCluster Size Distribution:")
    print(f"  Singletons (1 record):    {singleton:>8,} ({100*singleton/len(cluster_sizes):.1f}%)")
    print(f"  Pairs (2 records):        {size_2:>8,} ({100*size_2/len(cluster_sizes):.1f}%)")
    print(f"  Small (3-5 records):      {size_3_5:>8,} ({100*size_3_5/len(cluster_sizes):.1f}%)")
    print(f"  Medium (6-10 records):    {size_6_10:>8,} ({100*size_6_10/len(cluster_sizes):.1f}%)")
    print(f"  Large (>10 records):      {size_large:>8,} ({100*size_large/len(cluster_sizes):.1f}%)")
    
    print(f"\nLargest cluster: {cluster_sizes.max()} records")
    print(f"Average cluster size: {cluster_sizes.mean():.2f}")
    print(f"Median cluster size: {cluster_sizes.median():.1f}")
else:
    print("No clusters found. Run 3_inference.ipynb with clustering enabled.")


In [ ]:
# Inspect largest clusters (may indicate problematic matches)
if clusters_df is not None and len(cluster_sizes) > 0:
    print("LARGEST CLUSTERS")
    print("=" * 50)
    print("(Large clusters may indicate over-matching)")
    
    largest_cluster_ids = cluster_sizes.nlargest(5).index
    
    for cluster_id in largest_cluster_ids:
        cluster_records = clusters_df[clusters_df['cluster_id'] == cluster_id]
        size = len(cluster_records)
        
        print(f"\nCluster {cluster_id} ({size} records):")
        
        # Show names if available
        if 'name' in cluster_records.columns:
            names = cluster_records['name'].head(5).tolist()
        elif 'unique_id' in cluster_records.columns and dim_org_names:
            names = [dim_org_names.get(uid, uid) for uid in cluster_records['unique_id'].head(5)]
        else:
            names = cluster_records['unique_id'].head(5).tolist()
        
        for name in names:
            print(f"  - {str(name)[:70]}")
        if size > 5:
            print(f"  ... and {size - 5} more")


---
## 3. Splink Model Visualizations


In [ ]:
# Load model for visualizations
import json
from splink import Linker, DuckDBAPI
from utils import load_json

model_json = load_json(config.paths.MODEL_FILE, "Trained model")

if model_json is not None:
    # Remove training metadata (not a Splink setting)
    model_settings = {k: v for k, v in model_json.items() if k != 'training_metadata'}
    
    # Initialize linker for visualizations
    linker = Linker([dim_org_df], model_settings, db_api=DuckDBAPI())
    print("Linker initialized for visualizations")
else:
    linker = None
    print("Model not found - visualizations unavailable")


In [ ]:
# Match weights chart - shows contribution of each comparison
if linker is not None:
    print("MATCH WEIGHTS CHART")
    print("=" * 50)
    print("Shows how each comparison contributes to match probability")
    print("Positive weights support match, negative weights oppose\n")
    
    linker.visualisations.match_weights_chart()


In [ ]:
# M/U probability chart - shows parameter estimates
if linker is not None:
    print("M AND U PROBABILITY CHART")
    print("=" * 50)
    print("M = probability of agreement given records match")
    print("U = probability of random agreement (non-match)\n")
    
    linker.visualisations.m_u_parameters_chart()


In [ ]:
# Waterfall chart for a sample prediction
# Shows how each comparison contributes to the final match probability
if linker is not None and len(predictions_df) > 0:
    print("WATERFALL CHART (Sample High-Confidence Match)")
    print("=" * 50)
    
    # Get a sample high-confidence prediction
    sample_pred = predictions_df[predictions_df['match_probability'] > 0.95].head(1)
    
    if len(sample_pred) > 0:
        print(f"Sample pair (prob={sample_pred['match_probability'].values[0]:.4f}):\n")
        
        # Convert to format linker expects
        linker.visualisations.waterfall_chart(sample_pred.to_dict('records'))


---
## 2. Match Rate by Source


In [ ]:
# Match rate by source table
print("MATCH RATE BY SOURCE TABLE")
print("=" * 70)

if 'source_table' in predictions_df.columns:
    source_stats = predictions_df.groupby('source_table').agg({
        'match_probability': ['count', 'mean', 'median', 'max']
    }).round(3)
    source_stats.columns = ['count', 'mean_prob', 'median_prob', 'max_prob']
    source_stats = source_stats.sort_values('count', ascending=False)
    
    print(source_stats.to_string())
else:
    print("No source_table column in predictions")


---
## 3. Confidence Distribution


In [ ]:
# Confidence tier distribution
print("CONFIDENCE TIER DISTRIBUTION")
print("=" * 50)

# Define tiers
predictions_df['confidence_tier'] = pd.cut(
    predictions_df['match_probability'],
    bins=[0, 0.5, 0.7, 0.85, 0.95, 1.0],
    labels=['<0.5 (Very Low)', '0.5-0.7 (Low)', '0.7-0.85 (Medium)', '0.85-0.95 (High)', '0.95-1.0 (Very High)']
)

tier_counts = predictions_df['confidence_tier'].value_counts().sort_index()

print("\nDistribution:")
for tier, count in tier_counts.items():
    pct = 100 * count / len(predictions_df)
    bar = '#' * int(pct / 2)
    print(f"  {tier:<25} | {bar:<50} {count:>8,} ({pct:5.1f}%)")


In [ ]:
# Basic statistics
print("\nMATCH PROBABILITY STATISTICS")
print("=" * 50)
print(predictions_df['match_probability'].describe())


---
## 4. Sample Matches at Each Tier


In [ ]:
def show_sample_matches(df, tier_name, n=5):
    """Display sample matches from a tier"""
    print(f"\n{tier_name}")
    print("-" * 70)
    
    sample = df.head(n)
    for i, (_, row) in enumerate(sample.iterrows()):
        left_id = row.get('unique_id_l', 'N/A')
        right_id = row.get('unique_id_r', 'N/A')
        prob = row.get('match_probability', 0)
        
        # Get names from lookup
        left_name = dim_org_names.get(left_id, left_id)
        right_name = dim_org_names.get(right_id, right_id)
        
        print(f"\n  Match {i+1} (prob={prob:.4f}):")
        print(f"    L: {str(left_name)[:65]}")
        print(f"    R: {str(right_name)[:65]}")


In [ ]:
# Very high confidence matches (>0.95)
very_high = predictions_df[predictions_df['match_probability'] > 0.95].sort_values('match_probability', ascending=False)
show_sample_matches(very_high, "VERY HIGH CONFIDENCE (>0.95)")


In [ ]:
# Medium confidence matches (0.7-0.85) - these may need review
medium = predictions_df[(predictions_df['match_probability'] >= 0.7) & (predictions_df['match_probability'] < 0.85)]
show_sample_matches(medium, "MEDIUM CONFIDENCE (0.7-0.85) - May Need Review")


In [ ]:
# Low confidence matches (0.5-0.7) - likely false positives
low = predictions_df[(predictions_df['match_probability'] >= 0.5) & (predictions_df['match_probability'] < 0.7)]
show_sample_matches(low, "LOW CONFIDENCE (0.5-0.7) - Likely False Positives")


---
## 5. Quality Analysis


---
## 5. Quality Analysis


In [ ]:
# Identify potential issues
print("QUALITY ANALYSIS")
print("=" * 50)

# Check for multiple matches to same dim_org record
if 'unique_id_l' in predictions_df.columns:
    match_counts = predictions_df.groupby('unique_id_l').size()
    multiple_matches = match_counts[match_counts > 1]
    
    print(f"\nRecords with multiple matches: {len(multiple_matches):,}")
    if len(multiple_matches) > 0:
        print(f"  Max matches to single record: {multiple_matches.max()}")
        print(f"\nTop records with most matches:")
        for uid, count in multiple_matches.nlargest(5).items():
            name = dim_org_names.get(uid, uid)
            print(f"  {count}x | {str(name)[:60]}")


---
## 6. Export Final Outputs


In [ ]:
# Create final output with different confidence levels
print("CREATING FINAL OUTPUTS")
print("=" * 50)

# High confidence matches only (production ready)
high_conf = predictions_df[predictions_df['match_probability'] >= config.matching.THRESHOLD_HIGH_CONFIDENCE].copy()

# Add human-readable names
if dim_org_names:
    high_conf['matched_org_name'] = high_conf['unique_id_l'].map(dim_org_names)

# Select final columns
output_cols = ['unique_id_r', 'unique_id_l', 'match_probability', 'source_table']
if 'matched_org_name' in high_conf.columns:
    output_cols.append('matched_org_name')

final_output = high_conf[output_cols].rename(columns={
    'unique_id_r': 'mismatched_id',
    'unique_id_l': 'dim_org_id',
    'match_probability': 'confidence'
})

print(f"High confidence matches: {len(final_output):,}")
print(f"\nSample output:")
display(final_output.head(10))


In [ ]:
# Save final outputs
output_path = config.paths.FINAL_MATCHES
final_output.to_csv(output_path, index=False)
print(f"\nFinal matches saved to: {output_path}")

# Also save as parquet for faster loading
parquet_path = output_path.replace('.csv', '.parquet')
final_output.to_parquet(parquet_path, index=False)
print(f"Parquet version: {parquet_path}")


In [ ]:
# Analysis summary
print("\n" + "=" * 70)
print("ANALYSIS COMPLETE")
print("=" * 70)

print(f"""
RESULTS SUMMARY:
  - Total predictions analyzed: {len(predictions_df):,}
  - High confidence (>0.95): {len(very_high):,}
  - Medium confidence (0.7-0.85): {len(medium):,}
  - Low confidence (0.5-0.7): {len(low):,}

QUALITY METRICS:
  - Records with multiple matches: {len(multiple_matches):,}

OUTPUT FILES:
  - Final matches (CSV): {config.paths.FINAL_MATCHES}
  - Final matches (Parquet): {parquet_path}

RECOMMENDATIONS:
  - High confidence matches can be used directly
  - Medium confidence matches should be reviewed
  - Low confidence matches are likely false positives
""")

print("Analysis complete!")
